# Samsung S25 YouTube Comments - Data Preprocessing

This notebook contains data preprocessing and analysis for Samsung S25 YouTube comments.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 2. Load Data

In [ ]:
# Load the CSV file
df = pd.read_csv('../samsung25.csv')
print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape}")

## 3. Initial Data Exploration

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Dataset information
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
# Check for duplicate comments
duplicates = df.duplicated(subset=['text']).sum()
print(f"Duplicate comments: {duplicates}")

duplicate_ids = df.duplicated(subset=['comment_id']).sum()
print(f"Duplicate comment IDs: {duplicate_ids}")

## 4. Basic Statistics

In [ ]:
print("=" * 70)
print("DATASET STATISTICS")
print("=" * 70)
print(f"Total Comments: {len(df)}")
print(f"Top-level Comments: {len(df[df['is_reply'] == False])}")
print(f"Replies: {len(df[df['is_reply'] == True])}")
print(f"Unique Authors: {df['author'].nunique()}")
print(f"Total Likes: {df['likes'].sum()}")
print(f"Average Likes: {df['likes'].mean():.2f}")
print(f"Median Likes: {df['likes'].median():.2f}")
print(f"Max Likes: {df['likes'].max()}")
print("=" * 70)

## 5. Data Cleaning

In [ ]:
# Remove duplicates if any
df_clean = df.drop_duplicates(subset=['comment_id'])
print(f"Removed {len(df) - len(df_clean)} duplicate comment IDs")

# Remove rows with missing text
df_clean = df_clean[df_clean['text'].notna()]
df_clean = df_clean[df_clean['text'].str.strip() != '']
print(f"Shape after cleaning: {df_clean.shape}")

## 6. Text Preprocessing

In [ ]:
# Add text length column
df_clean['text_length'] = df_clean['text'].str.len()
df_clean['word_count'] = df_clean['text'].str.split().str.len()

print(f"Average text length: {df_clean['text_length'].mean():.2f} characters")
print(f"Average word count: {df_clean['word_count'].mean():.2f} words")

In [ ]:
# Create a cleaned text column
def clean_text(text):
    if pd.isna(text):
        return ''
    # Convert to lowercase
    text = str(text).lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text

df_clean['text_cleaned'] = df_clean['text'].apply(clean_text)
print("Text cleaning completed!")

## 7. Visualizations

In [ ]:
# Likes distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(df_clean['likes'], bins=50, edgecolor='black')
axes[0].set_xlabel('Likes')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Likes')
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(df_clean['likes'])
axes[1].set_ylabel('Likes')
axes[1].set_title('Box Plot of Likes')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Top-level comments vs Replies
reply_counts = df_clean['is_reply'].value_counts()

plt.figure(figsize=(8, 6))
plt.pie(reply_counts.values, labels=['Top-level', 'Replies'], autopct='%1.1f%%', startangle=90)
plt.title('Top-level Comments vs Replies')
plt.show()

In [ ]:
# Text length distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(df_clean['text_length'], bins=50, edgecolor='black')
axes[0].set_xlabel('Text Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Text Length')
axes[0].grid(True, alpha=0.3)

axes[1].hist(df_clean['word_count'], bins=50, edgecolor='black', color='orange')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Word Count')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Top authors by comment count
top_authors = df_clean['author'].value_counts().head(10)

plt.figure(figsize=(12, 6))
top_authors.plot(kind='barh')
plt.xlabel('Number of Comments')
plt.ylabel('Author')
plt.title('Top 10 Most Active Commenters')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Top liked comments
top_liked = df_clean.nlargest(10, 'likes')[['author', 'text', 'likes']]
print("Top 10 Most Liked Comments:")
print("=" * 100)
for idx, row in top_liked.iterrows():
    print(f"\nAuthor: {row['author']}")
    print(f"Likes: {row['likes']}")
    print(f"Comment: {row['text'][:200]}..." if len(row['text']) > 200 else f"Comment: {row['text']}")
    print("-" * 100)

## 8. Save Preprocessed Data

In [ ]:
# Save cleaned data
output_file = '../samsung25_preprocessed.csv'
df_clean.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Preprocessed data saved to: {output_file}")
print(f"Final shape: {df_clean.shape}")

## 9. Summary

In [ ]:
print("=" * 70)
print("PREPROCESSING SUMMARY")
print("=" * 70)
print(f"Original dataset: {df.shape[0]} rows")
print(f"Cleaned dataset: {df_clean.shape[0]} rows")
print(f"Rows removed: {df.shape[0] - df_clean.shape[0]}")
print(f"Columns: {df_clean.shape[1]}")
print(f"\nNew columns added:")
print("  - text_length")
print("  - word_count")
print("  - text_cleaned")
print("=" * 70)